# 04. Advanced Features

This notebook demonstrates some of the more advanced features of the framework, including regularization, benchmarking, and model customization.

In [ ]:
import torch
import pytorch_lightning as pl
from pathlib import Path
from omegaconf import OmegaConf
from dataclasses import asdict

from data.factory import DatasetFactory
from net.common import Config
from net.e3gnn import E3GNN
from net.benchmark import BenchmarkCallback

## Regularization and Gradient Clipping

The `Config` dataclass allows you to easily configure L1/L2 regularization and gradient clipping.

In [ ]:
hp_reg = Config(
    l1_reg_coef=1e-5,
    l2_reg_coef=1e-4,
    grad_clip_val=0.5,
)

print("Config with regularization:")
print(hp_reg)

These hyperparameters are then used by the `E3GNN` model and the PyTorch Lightning `Trainer`.

## Benchmarking

The `BenchmarkCallback` can be used to profile the training process.

In [ ]:
# Create a DatasetFactory
script_dir = Path(__file__).parent.resolve()
factory = DatasetFactory()
factory.add_snapshot(
    script_dir / "../../data/big/silicon/900K/Si_DM",
    script_dir / "../../data/big/silicon/900K/info.txt",
)
train_ds, _, mapper = factory.create()

# Create a model
cfg = Config()
mock_cfg = OmegaConf.create(
    {"model": asdict(cfg), "training": {"lr": 1e-3}, "logging": {"safety_checks": False}}
)
model = E3GNN(mapper, train_ds.edge_types, mock_cfg)

# Create a BenchmarkCallback
benchmark_callback = BenchmarkCallback(verbosity=1)

# Create a Trainer with the callback
trainer = pl.Trainer(max_epochs=1, accelerator="cpu", devices=1, callbacks=[benchmark_callback])

# Run training
train_dl = torch.utils.data.DataLoader(train_ds, batch_size=1, collate_fn=lambda b: b[0])
trainer.fit(model, train_dl)

The benchmark results, including a detailed breakdown of timings, will be saved to a YAML file in the `benchmarks/` directory.

## Model Customization

The `Config` dataclass provides a simple way to customize the model architecture.

In [ ]:
hp_custom = Config(
    hidden_base_dim=128,
    l_max=3,
    num_layers_gnn=4,
    num_layers_matrix=2,
    head_depth=2,
)

print("Custom Config:")
print(hp_custom)

## Matrix Operations

The `BlockMatrix` class supports various operations, including transposition and rotation.

In [ ]:
from data.snapshot import Snapshot

# Load a snapshot
snapshot = Snapshot.from_openmx(
    script_dir / "../../data/big/silicon/900K/Si_DM",
    script_dir / "../../data/big/silicon/900K/info.txt",
)

# Transpose the density matrix
density = snapshot.density
transposed_density = density.transpose()

print("Original density['Si-Si'] shape:", density["Si-Si"].shape)
print("Transposed density['Si-Si'] shape:", transposed_density["Si-Si"].shape)

# Rotate the snapshot
rotation_matrix = torch.randn(3, 3)
rotated_snapshot = snapshot.rotate(rotation_matrix)

print("\nRotated snapshot energy:", rotated_snapshot.get_energy().item())